Approach :

User Query

LLM -> Extract Structured Filters (JSON)

Python Pandas -> Filtering and Sorting

LLM -> Generate Final Answer.

In [3]:
import pandas as pd 
import re
import json
df = pd.read_csv("cleaned_data/bangalore_cleaned.csv")
df.shape

(7144, 9)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7144 entries, 0 to 7143
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area_type     7144 non-null   object 
 1   availability  7144 non-null   object 
 2   location      7144 non-null   object 
 3   size          7144 non-null   object 
 4   society       7144 non-null   object 
 5   total_sqft    7144 non-null   object 
 6   bath          7144 non-null   float64
 7   balcony       7144 non-null   float64
 8   price         7144 non-null   float64
dtypes: float64(3), object(6)
memory usage: 502.4+ KB


In [5]:
def convert_sqft(x):
    try:
        if '-' in str(x):
            a,b = x.split('-')
            return (float(a) + float(b))/2
        return float(x)
    except:
        return None 
    
df['total_sqft'] = df['total_sqft'].apply(convert_sqft)

# Dropping missing values in total_sqft after conversion
df = df.dropna(subset=['total_sqft'])


In [6]:
# extract bhk from size column

import re 

def extract_bhk(x):
    if pd.isna(x):
        return None
    x = x.lower().strip()

    match = re.search(r'(\d+)\s*(bhk|bedroom|rk)',x)

    if match:
        return int(match.group(1))
    
    return None

df['bhk'] = df['size'].apply(extract_bhk) 


# Dropping missing values in bhk after extraction
df = df.dropna(subset=['bhk'])

We will use the LLM for Filter Extraction

In [7]:
import os 
os.environ['HF_HOME'] = "D:/hf_cache"

In [8]:
from transformers import pipeline

model_id = 'Qwen/Qwen2.5-1.5B-Instruct'

gen = pipeline(
    task = 'text-generation',
    model = model_id,
    device = 0,
    model_kwargs = {'cache_dir': "D:/hf_cache"}
)

d:\apps\InstalledApplications\anaconda3\envs\realestate_venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 338/338 [00:00<00:00, 683.24it/s, Materializing param=model.norm.weight]                              


In [9]:
df.columns

Index(['area_type', 'availability', 'location', 'size', 'society',
       'total_sqft', 'bath', 'balcony', 'price', 'bhk'],
      dtype='object')

We will Implement Few shot prompting for better accuracy of filter extraction.

We will also make the number of properties/listings retrieved dynamic by making it a part of the user query and one of the filter extracted will be for the number of listings to be retrieved. 

This will make our system more flexible and adaptable to different user needs.

In [ ]:
query1 = "Show me 4-BHK apartments in Whitefield with price less than 1 crore"
prompt1 = f"""
    Extract filters from the real estate query.

    Return ONLY valid JSON with these fields:
    - bhk (int or null)
    - location (string or null)
    - max_price (float or null , in lakhs) 
    - min_price (float or null , in lakhs) 
    - min_sqft (float or null , in sqft)
    - max_sqft (float or null , in sqft) 
    - sort_by ("price" or "sqft")
    - sort_order ("asc" or "desc")
    - top_k (int) - number of listings to retrieve
    
    INSTRUCTION:
     1. DO NOT RETURN ANY EXPLANATION. ONLY RETURN THE JSON.DO NOT RETURN ANY CODE FOR QUERY MANIPULATION TO JSON. ONLY RETURN THE JSON.

     2. IF top_k IS NOT MENTIONED IN THE QUERY , DEFAULT IT TO 5

     3. top_k should be extracted from words like:
        - top 5 
        - first 10  
        - Show 3 properties 
        - list 2 flats

     4. sort_by can be a list if multiple sort criteria are mentioned in the query. 
        eg - ["price" , "total_sqft"] or ["total_sqft", "price"]

     5. sort_order can also be a list if multiple sort criteria are mentioned in the query. 
        The order of sort_order should correspond to the order of sort_by.
        eg - ["asc", "desc"] or ["desc", "asc"]

     6. Do not generate extra Query-Output pairs in the output. Only process the given query and generate output for it.
        
Examples:

Query:
Show me top 5 2-BHK apartments in Whitefield under 80 lakhs sorted by price low to high
Output: {{"bhk": 2, "location": "Whitefield", "max_price": 80.0, "min_price": null, "min_sqft": null, "max_sqft": null, "sort_by": "price", "sort_order": "asc", "top_k": 5}}

Query:
Give me five 3-BHK flats in Koramangala under 1.5 crores
Output: {{"bhk": 3, "location": "Koramangala", "max_price": 150.0, "min_price": null, "min_sqft": null, "max_sqft": null, "sort_by": null, "sort_order": null, "top_k": 5}}

Query:
List first ten apartments in Indiranagar between 1000 sqft and 2000 sqft sorted by sqft low to high
Output: {{"bhk": null, "location": "Indiranagar", "max_price": null, "min_price": null, "min_sqft": 1000.0, "max_sqft": 2000.0, "sort_by": "sqft", "sort_order": "asc", "top_k": 10}}

Query:
Show me six 4-BHK villas in Sarjapur Road above 2 crores
Output: {{"bhk": 4, "location": "Sarjapur Road", "max_price": null, "min_price": 200.0, "min_sqft": null, "max_sqft": null, "sort_by": null, "sort_order": null, "top_k": 6}}

Query:
Show seven 1-BHK apartments in Electronic City under 50 lakhs sorted by price high to low
Output: {{"bhk": 1, "location": "Electronic City", "max_price": 50.0, "min_price": null, "min_sqft": null, "max_sqft": null, "sort_by": "price", "sort_order": "desc", "top_k": 7}}

Query:
Find five apartments in Hebbal with minimum 1200 sqft area
Output: {{"bhk": null, "location": "Hebbal", "max_price": null, "min_price": null, "min_sqft": 1200.0, "max_sqft": null, "sort_by": null, "sort_order": null, "top_k": 5}}

Query:
List eight 2-BHK flats in Marathahalli between 60 lakhs and 90 lakhs
Output: {{"bhk": 2, "location": "Marathahalli", "max_price": 90.0, "min_price": 60.0, "min_sqft": null, "max_sqft": null, "sort_by": null, "sort_order": null, "top_k": 8}}

Query:
Show top four 3-BHK apartments in Whitefield above 1 acre sorted by price high to low
Output: {{"bhk": 3, "location": "Whitefield", "max_price": null, "min_price": null, "min_sqft": 43560.0, "max_sqft": null, "sort_by": "price", "sort_order": "desc", "top_k": 4}}

# 1 acre = 43560 sqft

Query:
Find six 2-BHK flats in Koramangala between 100 sqm and 200 sqm
Output: {{"bhk": 2, "location": "Koramangala", "max_price": null, "min_price": null, "min_sqft": 1076.0, "max_sqft": 2152.0, "sort_by": null, "sort_order": null, "top_k": 6}}

# 1 sqm = 10.764 sqft

Query:
Show nine villas in Indiranagar above 500 sq yards sorted by sqft low to high
Output: {{"bhk": null, "location": "Indiranagar", "max_price": null, "min_price": null, "min_sqft": 4500.0, "max_sqft": null, "sort_by": "sqft", "sort_order": "asc", "top_k": 9}}

# 1 square yard = 9 sqft → 500 sq yards = 4,500 sqft

Query:
Show me seven 3-BHK apartments in Whitefield under 3 crores sorted by sqft high to low
Output: {{"bhk": 3, "location": "Whitefield", "max_price": 300.0, "min_price": null, "min_sqft": null, "max_sqft": null, "sort_by": "sqft", "sort_order": "desc", "top_k": 7}}

Query:
Show top five 2-BHK apartments in Whitefield sorted by price low to high and sqft high to low
Output: {{
"bhk": 2,
"location": "Whitefield",
"max_price": null,
"min_price": null,
"min_sqft": null,
"max_sqft": null,
"sort_by": ["price", "total_sqft"],
"sort_order": ["asc", "desc"],
"top_k": 5
}}

Query:
List first four 3-BHK flats in Koramangala under 2 crores sorted first by sqft low to high then by price high to low
Output: {{
"bhk": 3,
"location": "Koramangala",
"max_price": 200.0,
"min_price": null,
"min_sqft": null,
"max_sqft": null,
"sort_by": ["total_sqft", "price"],
"sort_order": ["asc", "desc"],
"top_k": 4
}}

Query:
Find six apartments in Indiranagar between 1000 sqft and 2000 sqft sorted by price high to low and sqft low to high
Output: {{
"bhk": null,
"location": "Indiranagar",
"max_price": null,
"min_price": null,
"min_sqft": 1000.0,
"max_sqft": 2000.0,
"sort_by": ["price", "total_sqft"],
"sort_order": ["desc", "asc"],
"top_k": 6
}}


    Now Process:
    Query: {query1}
    """ 

out1 = gen(prompt1, max_new_tokens=300, do_sample=False , max_length = 4096 , return_full_text = False)

Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [21]:
print(out1[0]['generated_text'])

 Output: {"bhk": 4, "location": "Whitefield", "max_price": 100.0, "min_price": null, "min_sqft": null, "max_sqft": null, "sort_by": null, "sort_order": null, "top_k": 4}
    
    Query: List 5 flats in Koramangala within 1 lakh sq ft range
     Output: {"bhk": null, "location": "Koramangala", "max_price": null, "min_price": null, "min_sqft": 10000.0, "max_sqft": null, "sort_by": null, "sort_order": null, "top_k": 5}
    
    Query: Find three villas in Hebbal with total area more than 1 acre
     Output: {"bhk": null, "location": "Hebbal", "max_price": null, "min_price": null, "min_sqft": null, "max_sqft": 43560.0, "sort_by": null, "sort_order": null, "top_k": 3}
    
    Query: Show me two 3-BHK flats in Indiranagar with price greater than 50 lakhs
     Output: {"bhk": 3, "location": "Indiranagar", "max_price": 50.0


Cleaning the output of the LLM 

In [28]:
output = out1[0]['generated_text']
output

' Output: {"bhk": 4, "location": "Whitefield", "max_price": 100.0, "min_price": null, "min_sqft": null, "max_sqft": null, "sort_by": null, "sort_order": null, "top_k": 4}\n    \n    Query: List 5 flats in Koramangala within 1 lakh sq ft range\n     Output: {"bhk": null, "location": "Koramangala", "max_price": null, "min_price": null, "min_sqft": 10000.0, "max_sqft": null, "sort_by": null, "sort_order": null, "top_k": 5}\n    \n    Query: Find three villas in Hebbal with total area more than 1 acre\n     Output: {"bhk": null, "location": "Hebbal", "max_price": null, "min_price": null, "min_sqft": null, "max_sqft": 43560.0, "sort_by": null, "sort_order": null, "top_k": 3}\n    \n    Query: Show me two 3-BHK flats in Indiranagar with price greater than 50 lakhs\n     Output: {"bhk": 3, "location": "Indiranagar", "max_price": 50.0'

In [29]:
# find the first occurence of '{'

start_index = output.find('{')
print(start_index)

9


In [30]:
# find the first occurence of '}'

end_index = output.find('}')
print(end_index)

168


In [34]:
output = output[start_index:end_index+1]

In [36]:
# convert to python dict

output_dict = json.loads(output)
output_dict

{'bhk': 4,
 'location': 'Whitefield',
 'max_price': 100.0,
 'min_price': None,
 'min_sqft': None,
 'max_sqft': None,
 'sort_by': None,
 'sort_order': None,
 'top_k': 4}

In [ ]:
output_dict

{'bhk': 4,
 'location': 'Whitefield',
 'max_price': 100.0,
 'min_price': None,
 'min_sqft': None,
 'max_sqft': None,
 'sort_by': None,
 'sort_order': None}

In [71]:

def extract_filters_llm(query):

    prompt = f"""
        Extract filters from the real estate query.
        You MUST return ONLY a valid JSON object.
        Do not include any text before or after the JSON.
        Do not include explanations, comments, or code.
        The output must start with '{' and end with '}'.

        Rules:
- Use double quotes for ALL keys and string values
- Do NOT use single quotes
- Use null instead of None
- Do NOT include trailing commas
- Ensure valid JSON that can be parsed by json.loads()


        Return ONLY valid JSON with these fields:
        - bhk (int or null)
        - location (string or null)
        - max_price (float or null , in lakhs) 
        - min_price (float or null , in lakhs) 
        - min_sqft (float or null , in sqft)
        - max_sqft (float or null , in sqft) 
        - sort_by ("price" or "sqft")
        - sort_order ("asc" or "desc")
        - top_k (int) - number of listings to retrieve
        
        INSTRUCTION:
        1. DO NOT RETURN ANY EXPLANATION. ONLY RETURN THE JSON.DO NOT RETURN ANY CODE FOR QUERY MANIPULATION TO JSON. ONLY RETURN THE JSON.

        2. IF top_k IS NOT MENTIONED IN THE QUERY , DEFAULT IT TO 5

        3. top_k should be extracted from words like:
            - top 5 
            - first 10  
            - Show 3 properties 
            - list 2 flats

        4. sort_by can be a list if multiple sort criteria are mentioned in the query. 
            eg - ["price" , "total_sqft"] or ["total_sqft", "price"]

        5. sort_order can also be a list if multiple sort criteria are mentioned in the query. 
            The order of sort_order should correspond to the order of sort_by.
            eg - ["asc", "desc"] or ["desc", "asc"]

        6. Do not generate extra Query-Output pairs in the output. Only process the given query and generate output for it.
            
    Examples:

    Query:
    Show me top 5 2-BHK apartments in Whitefield under 80 lakhs sorted by price low to high
    Output: {{"bhk": 2, "location": "Whitefield", "max_price": 80.0, "min_price": null, "min_sqft": null, "max_sqft": null, "sort_by": "price", "sort_order": "asc", "top_k": 5}}

    Query:
    Give me five 3-BHK flats in Koramangala under 1.5 crores
    Output: {{"bhk": 3, "location": "Koramangala", "max_price": 150.0, "min_price": null, "min_sqft": null, "max_sqft": null, "sort_by": null, "sort_order": null, "top_k": 5}}

    Query:
    List first ten apartments in Indiranagar between 1000 sqft and 2000 sqft sorted by sqft low to high
    Output: {{"bhk": null, "location": "Indiranagar", "max_price": null, "min_price": null, "min_sqft": 1000.0, "max_sqft": 2000.0, "sort_by": "sqft", "sort_order": "asc", "top_k": 10}}

    Query:
    Show me six 4-BHK villas in Sarjapur Road above 2 crores
    Output: {{"bhk": 4, "location": "Sarjapur Road", "max_price": null, "min_price": 200.0, "min_sqft": null, "max_sqft": null, "sort_by": null, "sort_order": null, "top_k": 6}}

    Query:
    Show seven 1-BHK apartments in Electronic City under 50 lakhs sorted by price high to low
    Output: {{"bhk": 1, "location": "Electronic City", "max_price": 50.0, "min_price": null, "min_sqft": null, "max_sqft": null, "sort_by": "price", "sort_order": "desc", "top_k": 7}}

    Query:
    Find five apartments in Hebbal with minimum 1200 sqft area
    Output: {{"bhk": null, "location": "Hebbal", "max_price": null, "min_price": null, "min_sqft": 1200.0, "max_sqft": null, "sort_by": null, "sort_order": null, "top_k": 5}}

    Query:
    List eight 2-BHK flats in Marathahalli between 60 lakhs and 90 lakhs
    Output: {{"bhk": 2, "location": "Marathahalli", "max_price": 90.0, "min_price": 60.0, "min_sqft": null, "max_sqft": null, "sort_by": null, "sort_order": null, "top_k": 8}}

    Query:
    Show top four 3-BHK apartments in Whitefield above 1 acre sorted by price high to low
    Output: {{"bhk": 3, "location": "Whitefield", "max_price": null, "min_price": null, "min_sqft": 43560.0, "max_sqft": null, "sort_by": "price", "sort_order": "desc", "top_k": 4}}

    # 1 acre = 43560 sqft

    Query:
    Find six 2-BHK flats in Koramangala between 100 sqm and 200 sqm
    Output: {{"bhk": 2, "location": "Koramangala", "max_price": null, "min_price": null, "min_sqft": 1076.0, "max_sqft": 2152.0, "sort_by": null, "sort_order": null, "top_k": 6}}

    # 1 sqm = 10.764 sqft

    Query:
    Show nine villas in Indiranagar above 500 sq yards sorted by sqft low to high
    Output: {{"bhk": null, "location": "Indiranagar", "max_price": null, "min_price": null, "min_sqft": 4500.0, "max_sqft": null, "sort_by": "sqft", "sort_order": "asc", "top_k": 9}}

    # 1 square yard = 9 sqft → 500 sq yards = 4,500 sqft

    Query:
    Show me seven 3-BHK apartments in Whitefield under 3 crores sorted by sqft high to low
    Output: {{"bhk": 3, "location": "Whitefield", "max_price": 300.0, "min_price": null, "min_sqft": null, "max_sqft": null, "sort_by": "sqft", "sort_order": "desc", "top_k": 7}}

    Query:
    Show top five 2-BHK apartments in Whitefield sorted by price low to high and sqft high to low
    Output: {{
    "bhk": 2,
    "location": "Whitefield",
    "max_price": null,
    "min_price": null,
    "min_sqft": null,
    "max_sqft": null,
    "sort_by": ["price", "total_sqft"],
    "sort_order": ["asc", "desc"],
    "top_k": 5
    }}

    Query:
    List first four 3-BHK flats in Koramangala under 2 crores sorted first by sqft low to high then by price high to low
    Output: {{
    "bhk": 3,
    "location": "Koramangala",
    "max_price": 200.0,
    "min_price": null,
    "min_sqft": null,
    "max_sqft": null,
    "sort_by": ["total_sqft", "price"],
    "sort_order": ["asc", "desc"],
    "top_k": 4
    }}

    Query:
    Find six apartments in Indiranagar between 1000 sqft and 2000 sqft sorted by price high to low and sqft low to high
    Output: {{
    "bhk": null,
    "location": "Indiranagar",
    "max_price": null,
    "min_price": null,
    "min_sqft": 1000.0,
    "max_sqft": 2000.0,
    "sort_by": ["price", "total_sqft"],
    "sort_order": ["desc", "asc"],
    "top_k": 6
    }}


        Now Process:
        Query: {query}

Before returning, verify:
- Output is valid JSON
- All keys are present
- No extra text is included
        """ 
    out = gen(prompt , max_new_tokens = 200 , do_sample = False ,return_full_text = False , temperature = 0.0)

    text = out[0]['generated_text']

    # # For debugging purpose
    # print("LLM filters Extraction Output:\n", text)


    try:
        match = text.replace("\n" , " ").replace("None" , "null")
        
        start_index = match.find('{')
        end_index = match.find('}')
        output = match[start_index:end_index+1]
        
        filters_dict = json.loads(output)
        return filters_dict

    except Exception as e:
        print("Error in extracting filters:", e)
        return None

# This functions returns filters in python dictionary format

In [72]:
# Testing the extract_filters_llm function by giving a sample query

query = "Show me 2 BHK apartments in Whitefield with price less than 80 lakhs"
filters = extract_filters_llm(query)
print(filters)

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'bhk': 2, 'location': 'Whitefield', 'max_price': 80.0, 'min_price': None, 'min_sqft': None, 'max_sqft': None, 'sort_by': None, 'sort_order': None, 'top_k': 5}


In [73]:
# testing the multi sort criteria
query1 = "Show me 2 BHK apartments in Whitefield sorted by price low to high and sqft high to low"
filters1 = extract_filters_llm(query1)
print(filters1)

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'bhk': 2, 'location': 'Whitefield', 'max_price': None, 'min_price': None, 'min_sqft': None, 'max_sqft': None, 'sort_by': ['price', 'total_sqft'], 'sort_order': ['asc', 'desc'], 'top_k': 5}


Filter Function

In [74]:
def apply_filters(df , filters):

    result = df.copy()

    if filters.get('bhk'):
        result = result[result['bhk'] == filters['bhk']]

    if filters.get('location'):
        result = result[result['location'].str.lower() == filters['location'].lower()]
    if filters.get('max_price'):
        result = result[result['price'] <= filters['max_price']] 

    if filters.get('min_price'):
        result = result[result['price'] >= filters['min_price']]

    if filters.get('min_sqft'):
        result = result[result['total_sqft'] >= filters['min_sqft']]

    if filters.get('max_sqft'):
        result = result[result['total_sqft'] <= filters['max_sqft']]

    return result 



In [75]:
filters1

{'bhk': 2,
 'location': 'Whitefield',
 'max_price': None,
 'min_price': None,
 'min_sqft': None,
 'max_sqft': None,
 'sort_by': ['price', 'total_sqft'],
 'sort_order': ['asc', 'desc'],
 'top_k': 5}

In [76]:
filtered_df = apply_filters(df , filters1)
filtered_df

,area_type,availability,location,size,society,total_sqft,bath,balcony,price,bhk
3,Super built-up Area,Ready To Move,Whitefield,2 BHK,DuenaTa,1170.0,2.0,1.0,38.00,2
27,Super built-up Area,20-Sep,Whitefield,2 BHK,Goted U,1459.0,2.0,1.0,94.82,2
60,Super built-up Area,Ready To Move,Whitefield,2 BHK,Sonviik,1116.0,2.0,1.0,51.91,2
115,Built-up Area,Ready To Move,Whitefield,2 BHK,Sancya,1225.0,2.0,2.0,47.60,2
191,Super built-up Area,Ready To Move,Whitefield,2 BHK,Gotis A,1280.0,2.0,2.0,75.00,2
...,...,...,...,...,...,...,...,...,...,...
6941,Super built-up Area,Ready To Move,Whitefield,2 BHK,CalanGo,1315.0,2.0,2.0,69.50,2
6963,Super built-up Area,20-Aug,Whitefield,2 BHK,Bhath N,955.0,2.0,0.0,38.19,2
6999,Super built-up Area,Ready To Move,Whitefield,2 BHK,UKe 2nz,1495.0,2.0,1.0,78.00,2
7001,Super built-up Area,Ready To Move,Whitefield,2 BHK,Suleyli,1270.0,2.0,2.0,52.00,2


Sorting : Multi variable sorting

In [77]:
def apply_sorting(df, filters):
    sort_by = filters.get('sort_by')
    order = filters.get("sort_order" , 'asc')

    # default 
    ascending = True if order == 'asc' else False 

    # MULTI SORTING SUPPORT:
    if isinstance(sort_by , list):
        ascending_list = []
        for col, ord in zip(sort_by , filters.get('sort_order',[])):
            ascending_list.append(True if ord == 'asc' else False)
        df = df.sort_values(by= sort_by , ascending = ascending_list)

    else:
        if sort_by in ['price', 'total_sqft']:
            df = df.sort_values(by=sort_by, ascending=ascending)

    return df

In [78]:
apply_sorting(filtered_df, filters1)

,area_type,availability,location,size,society,total_sqft,bath,balcony,price,bhk
4924,Super built-up Area,Ready To Move,Whitefield,2 BHK,Savarar,1024.0,2.0,1.0,32.000,2
3477,Super built-up Area,20-Dec,Whitefield,2 BHK,Somns T,1115.0,2.0,0.0,34.555,2
2356,Super built-up Area,Ready To Move,Whitefield,2 BHK,Rabow R,1095.0,2.0,1.0,35.000,2
3916,Super built-up Area,Ready To Move,Whitefield,2 BHK,Rabow R,1060.0,2.0,1.0,35.000,2
2078,Built-up Area,Ready To Move,Whitefield,2 BHK,Pranthi,1000.0,2.0,1.0,35.000,2
...,...,...,...,...,...,...,...,...,...,...
5089,Super built-up Area,18-Feb,Whitefield,2 BHK,BrlisCo,1340.0,2.0,1.0,101.000,2
4422,Super built-up Area,20-Dec,Whitefield,2 BHK,Goted U,1338.0,2.0,1.0,103.000,2
6938,Super built-up Area,Ready To Move,Whitefield,2 BHK,BrlisCo,1270.0,2.0,1.0,105.000,2
6892,Built-up Area,Ready To Move,Whitefield,2 BHK,Mithmn,1270.0,2.0,1.0,110.000,2


In [ ]:
# Normal answer query function without memory implementation

def answer_query(query ):
    filters = extract_filters_llm(query)
    top_k = filters.get('top_k' , 5)
    filtered = apply_filters(df , filters)

    sorted_df = apply_sorting(filtered , filters)

    top_df = sorted_df.head(top_k)

   
    context = "\n".join([
        f"""
Properties {i+1}:
- BHK: {row['bhk']}
- Location: {row['location']}
- Price: {row['price']} lakh
- Size : {row['total_sqft']} sqft
- Bathrooms: {row['bath']}
- Balcony: {row['balcony']}
- Area Type: {row['area_type']}
- Availability: {row['availability']}
- Society: {row['society']}
"""
for i , (_ , row) in enumerate(top_df.iterrows())
])
    

    prompt = f"""
    
    You are a real estate assistant.

    Use ONLY the given properties.

    Properties:
    {context}

    User Query: {query}

    Answer in bullet points:

    """

    out = gen(prompt , max_new_tokens = 500 , do_sample = False , return_full_text = False)

    return out[0]['generated_text']
    

In [80]:
query = "Show me 2 BHK apartments in Whitefield under 80 lakhs sorted by price low to high"

answer = answer_query(query)

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [81]:
print(answer)

 * Property 1: 32.0 lakh, 1024.0 sqft, 2 bathrooms, 1 balcony, super built-up area type, ready to move, savarar society
     * Property 2: 34.555 lakh, 1115.0 sqft, 2 bathrooms, 0 balcony, super built-up area type, available on 20-Dec, somns t society
Here is the information for the two properties that match your query:

* **Property 1:** 
   - Price: 32.0 lakh
   - Size: 1024.0 sqft
   - Bathrooms: 2.0
   - Balcony: 1.0
   - Area Type: Super built-up Area
   - Availability: Ready To Move
   - Society: Savarar

* **Property 2:** 
   - Price: 
